# N091 · 函数图、环与入树

**目标：** 利用每点至多一条出边分离环和链。

**先修：** N082, N083, N088。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 入度剥离；访问时间戳；环长；入树最长链；二元环特例。

**配套讲解来源：** [同名逐章意见](../../comment/091_functional_graphs.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

**函数图（functional graph）**是一种特殊的有向图：每个点**至多一条出边**。它通常用一个数组表示——`edges[i]` 就是点 i 唯一的出边指向谁（-1 表示没有出边）。因为每人只有一条出路，你从任何点出发一路走下去，结局只有两种：要么走到一条边的尽头终止，要么**必然掉进一个环**里转圈。这种"链 + 环"的简单结构让我们能用很轻的算法解决两个实际问题：

**问题一（LeetCode 2360 找最长环）**：整张图里最长的环有多长？没有环就返回 -1。比如 `edges=[3,3,4,2,3]`（0→3、1→3、2→4、3→2、4→3）：3→2→4→3 构成长度 3 的环，答案就是 **3**。

**问题二（LeetCode 2127 会议邀请）**：n 个员工每人恰有一个"最想挨着坐"的人，大家坐成一个圆桌，要求每个人身边（左右任一侧）必须坐着自己喜欢的人。问最多能邀请多少人？比如 `favorite=[1,0,3,2]`：0 和 1 互相喜欢、2 和 3 互相喜欢，两张双人小桌拼起来一共 **4** 人。

再看本章 “运行示例”部分 的例子：`favorite=[1,0,1,2,5,4]`。

- 0↔1 是一个互相喜欢的对（二元环），2 喜欢 1、3 喜欢 2 形成一条挂上去的链；4↔5 又是一对。
- 圆桌答案：桌子一 {3,2,0,1}（3-2-0-1 围一圈，每人旁边都有喜欢的人），桌子二 {4,5}，合计 **6** 人——代码输出正是"可邀请人数 6"。

这个例子揭示了本章最微妙的知识点：**互相喜欢的两人组（二元环）可以挂链、还可以多组拼接**，而长度 ≥3 的环什么链都挂不上，只能整环单独成桌。

## 2. 基础知识：先读懂这些词

- **函数图（functional graph）**：每个点出度 ≤ 1 的有向图。名字来自"函数"的定义——每个输入（点）唯一对应一个输出（出边目标）。
- **"rho"形状（ρ 形）**：从一点出发的路径像希腊字母 ρ：先走一段"尾巴"（链），然后进入环绕圈。函数图里每个点出发的路径都是 ρ 形或纯链。
- **入度 / 出度**：指向该点的边数 / 从该点出发的边数。函数图里出度最多 1，入度不限。
- **入度剥离（拓扑剥离）**：不断删掉"入度变成 0"的点（就像拓扑排序）。在函数图上做这件事，会把所有"链"的部分一层层剥掉，**剩下的恰好全是环上的点**——因为环上的点入度永远 ≥1（环里前一个点始终指向它），剥不掉。
- **depth（最长入链长度）**：剥离过程中顺便维护的量：depth[v] 记录"从外面挂进 v 的最长链有几个点"。初始每个点为 1（自己算一个）。
- **访问时间戳（position）**：`longest_cycle` 里的技巧——沿路径走时给每个点记"这是本次路径的第几步"；如果走到了一个本次路径上已经出现过的点，两个序号相减就是环长。
- **旧路径 vs 本次路径**：判断环的关键是区分"这个点属于**本次**走的路径"（position 里有记录）还是"以前某个起点走过的**旧**路径"（visited 全局标记）。撞上旧路径不会再产生新环（那个环早被算过了），撞上本次路径才会。
- **二元环特例**：长度恰为 2 的环（A↔B）。座位问题里它的两端各有一个空邻座，所以左右各能挂一条入链；而且多个二元环桌子可以拼在一起（每桌都能自成圆桌）。长度 ≥3 的环：每个点的两个邻座都被环内前后两人占了，一条链也挂不上。
- **契约边界（“正确性与复杂度”部分 声明）**：`longest_cycle` 允许 -1 终点和教学用自环；`maximum_invitations` 要求 n≥2 且 favorite[i]≠i（没人喜欢自己），这符合座位问题的官方设定。

## 3. 思路推导：从小例子开始

### 最长环（longest_cycle）

- **Step 1**：全局 visited 数组防止重复走旧路径；answer 初始 -1（"没找到环"的答案）。
- **Step 2**：每个起点 s 单独建一个空的 position 字典，沿出边一路走，给没访问过的点记 visited 并写 position[点]=第几步。
- **Step 3**：停下只有两种原因——u 是 -1（路走到头）或 u 已 visited。此时若 u 在**本次**的 position 里，环长 = 总步数 − position[u]。
- **Step 4**：所有环长取最大。

**手算演示**（`edges=[3,3,4,2,3]`）：

- s=0：position={}。走 0（记 pos 0）→ 3（pos 1）→ 2（pos 2）→ 4（pos 3）→ 下一步 u=edges[4]=3，3 已 visited，循环停。u=3 在 position 且 position[3]=1 → 环长 = 4 − 1 = **3**。answer=3。
- s=1：position={}。走 1（pos 0）→ u=edges[1]=3 已 visited，停。3 不在本次 position（只有 {1:0}）→ 没有新环。（3、2、4 组成的环 s=0 时已算过，不能重复计。）
- s=2、3、4：全部 visited，直接跳过。
- 返回 3。

### 会议邀请（maximum_invitations）

- **Step 1**：统计每个点的入度 degree；所有入度 0 的点入队。
- **Step 2（入度剥离）**：队列里弹出 u，它指向 v=favorite[u]。用 depth[u]+1 尝试抬高 depth[v]（链经过 u 延伸到 v）；v 的入度减 1，减到 0 说明再没人从外面挂 v 了，v 也入队等待剥离。这个循环结束后，链全部被消化，degree>0 的点恰好都在环上。
- **Step 3**：遍历每个还在环上的点，沿 favorite 走一圈收集 cycle（seen 集合防止环之间重复处理——函数图里环与环互不相交）。
- **Step 4**：环长为 2 → 这是一张"双人桌"，它能带的人 = depth[两人] 之和（两条入链 + 桌上两人本身），**累加**进 paired（多桌可拼）；环长 ≥3 → 只能整环成桌，长度和 best 取最大。
- **Step 5**：答案 = max(paired, best)。两种结构（多个二元环拼桌 vs 单个长环）互斥，取大。

**手算演示**（“运行示例”部分 的 `favorite=[1,0,1,2,5,4]`）：

- 入度：0 被 1 喜欢（1），1 被 0 和 2 喜欢（2），2 被 3 喜欢（1），3 没人喜欢（0），4 被 5 喜欢（1），5 被 4 喜欢（1）。degree=[1,2,1,0,1,1]。
- 剥离：入度 0 的只有 3，出队。3 指向 2：depth[2]=max(1, depth[3]+1)=2，degree[2] 减到 0，2 入队。2 指向 1：depth[1]=max(1, 2+1)=**3**，degree[1] 从 2 减到 1（还有 0 指向它），不减到 0 不入队。队空。此时 depth=[1,3,2,1,1,1]，degree=[1,1,0,0,1,1]——degree>0 的 0、1、4、5 正是两个环的点。
- 收环：s=0 走出 cycle=[0,1]，长度 2 → paired += depth[0]+depth[1] = 1+3 = **4**（3、2、1、0 这一桌四人）。s=4 走出 cycle=[4,5]，长度 2 → paired += 1+1 = 2，累计 **6**。
- 没有长度 ≥3 的环，best=0。答案 max(6,0)=**6**，与 “运行示例”部分 输出一致。

## 4. 核心代码：longest_cycle

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def longest_cycle(edges):
    visited = [False] * len(edges)
    answer = -1
    for s in range(len(edges)):
        position = {}
        u = s
        while u != -1 and (not visited[u]):
            visited[u] = True
            position[u] = len(position)
            u = edges[u]
        if u in position:
            answer = max(answer, len(position) - position[u])
    return answer
```

### 逐段读懂代码

### 函数 1：`longest_cycle(edges)`

```python
from collections import deque

def longest_cycle(edges):
    visited = [False] * len(edges)
    answer = -1
    for s in range(len(edges)):
        position = {}
        u = s
        while u != -1 and (not visited[u]):
            visited[u] = True
            position[u] = len(position)
            u = edges[u]
        if u in position:
            answer = max(answer, len(position) - position[u])
    return answer
```

- 第 3 行：`visited` 是**跨起点共享**的全局标记（旧路径标记）；`answer=-1` 是"无环"的默认答案。
- 第 5 行：每个起点开一个**新的** position 字典（本次路径标记）。两个标记系统各司其职。
- 第 6 行的循环条件一句话挡住两种停机：`u!=-1` 挡"路走到尽头"（LeetCode 2360 允许 -1 表示无出边），`not visited[u]` 挡"撞上了走过的点"（可能是本次路径 = 找到环，也可能是旧路径 = 没有新环）。
- 第 7 行循环体：标记 visited、记录 position[u]（用 `len(position)` 当步数计数器，省一个变量）、沿着唯一的出边 `u=edges[u]` 挪一步。
- 第 8 行：循环退出后 u 若在**本次** position 里，说明从 position[u] 那一步开始绕回了本次路径——环长就是"总步数 − 进入环的第一步"。u 若是 -1 或旧路径的点，`u in position` 为 False，不做贡献。
- 第 9 行返回最大环长。

### 函数 2：`maximum_invitations(favorite)`

```python
def maximum_invitations(favorite):
    n = len(favorite)
    degree = [0] * n
    depth = [1] * n
    for v in favorite:
        degree[v] += 1
    q = deque((i for i, d in enumerate(degree) if d == 0))
```

- 第 2 行：degree 是入度表；depth 全部初始化为 1——每个点"自己一个人也算一条长度 1 的链"。
- 第 3 行一句统计入度：每条 i→favorite[i] 的边让终点入度加 1。
- 第 4 行：把所有"没人喜欢的人"（入度 0）装进队列，作为剥离的起点。`deque(...)` 直接从生成器构造。

```python
while q:
        u=q.popleft(); v=favorite[u]; depth[v]=max(depth[v],depth[u]+1); degree[v]-=1
        if degree[v]==0: q.append(v)
```

- 剥离主循环：u 是一个"外面再无人挂它"的点，它唯一指向 v。`depth[v]=max(depth[v],depth[u]+1)` 把"经过 u 到 v 的链长"和"v 已知的最长链"取大——多个链条汇入 v 时保留最长的那条。
- `degree[v]-=1` 消掉 u 贡献的那条入边；减到 0 说明 v 的所有外部来源都处理完了，v 也变成可剥离点入队。注意环上的点入度永远减不到 0（环内前驱的那条边不经过本循环，因为环上的点入度由环内点贡献且环点永不入队），所以循环自然只在链上传播。

```python
paired=best=0; seen=set()
    for s in range(n):
        if not degree[s] or s in seen: continue
        cycle=[]; u=s
        while u not in seen: seen.add(u); cycle.append(u); u=favorite[u]
        if len(cycle)==2: paired+=sum(depth[v] for v in cycle)
        else: best=max(best,len(cycle))
    return max(paired,best)
```

- 第 1 行两个累加器：paired 收集"所有二元环桌子能带的总人数"（可拼接），best 收集"最长的单个大环"（互斥方案）。
- 第 3 行过滤起点：`not degree[s]` 挑掉剥离后入度已归零的点（链上的点，不是环）；`s in seen` 跳过已收过的环。
- 第 4–5 行沿 favorite 走一圈收环：seen 全局去重保证每个环只收一次（走到 seen 里已有的点即闭合）。
- 第 6 行二元环分支：`paired+=sum(depth[v] for v in cycle)` 把两条入链的长度都加上——二元环两端各有一个空邻座，两条链都可用；多桌用 += 累加（题目允许多桌拼接）。
- 第 7 行长环分支：环长直接和 best 取最大——长度 ≥3 的环挂不了任何链。
- 第 8 行：两种结构答案取大。

## 5. 分段实现：按顺序运行

**本章接口：** `longest_cycle(edges)`、`maximum_invitations(favorite)`

### 导入本章使用的库

In [1]:
from collections import deque

### longest_cycle

In [2]:
def longest_cycle(edges):
    visited = [False] * len(edges)
    answer = -1
    for s in range(len(edges)):
        position = {}
        u = s
        while u != -1 and (not visited[u]):
            visited[u] = True
            position[u] = len(position)
            u = edges[u]
        if u in position:
            answer = max(answer, len(position) - position[u])
    return answer

### maximum_invitations

In [3]:
def maximum_invitations(favorite):
    n = len(favorite)
    degree = [0] * n
    depth = [1] * n
    for v in favorite:
        degree[v] += 1
    q = deque((i for i, d in enumerate(degree) if d == 0))
    while q:
        u = q.popleft()
        v = favorite[u]
        depth[v] = max(depth[v], depth[u] + 1)
        degree[v] -= 1
        if degree[v] == 0:
            q.append(v)
    paired = best = 0
    seen = set()
    for s in range(n):
        if not degree[s] or s in seen:
            continue
        cycle = []
        u = s
        while u not in seen:
            seen.add(u)
            cycle.append(u)
            u = favorite[u]
        if len(cycle) == 2:
            paired += sum((depth[v] for v in cycle))
        else:
            best = max(best, len(cycle))
    return max(paired, best)

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

favorite=[1,0,1,2,5,4]; print('可邀请人数',maximum_invitations(favorite))
show_table(['员工','喜欢的人'],list(enumerate(favorite)))

可邀请人数 6


员工,喜欢的人
0,1
1,0
2,1
3,2
4,5
5,4


## 7. 正确性、适用条件与复杂度

入度剥离消除所有树枝，depth[v]保留到达v的最长链长；残余顶点恰为环。二元环两端各有一个可用邻座，因此可以接链，其余环每个点的两个邻座均已占满。取所有二元环链之和与单个长环最大值，即覆盖所有可行结构。

**代价：** O(n)时间与空间。longest_cycle允许−1和教学自环；maximum_invitations要求n≥2且favorite[i]≠i，符合本章座位契约。

### 展开理解

**为什么入度剥离剩下的恰好是环？** 我们反过来想：一个点不在任何环上，说明从它出发沿出边走最终会终止或进入别的环，它自己处在"链"上。链的最外端一定是入度 0 的点（没人指向它），剥离从那里开始；删掉它之后，它指向的点入度减一——如果那个点也只被链上的点指着，入度迟早归零，也被剥掉。一层层剥下去，所有链上的点都会被消灭。而环上的每个点至少有一条入边来自环内的前一个点，这条边永远剥不掉，入度永远 ≥1。所以剥完之后 degree>0 的点不多不少正好是所有环上的点。剥离过程还顺手把 depth 算准了：depth[v] 是各条汇入 v 的链中最长者，因为每个 u 出队时 depth[u] 已经是它的最终值（u 之前已经吸收完它上游的所有链）。

**为什么二元环能挂链、长环不能？** 坐圆桌时每人左右两个邻座。二元环 A↔B：A 的左右两侧，一侧坐 B（环内占用），另一侧还空着，可以坐"最喜欢 A 的那条链"的末梢；B 同理——所以两端各挂一条链，桌长 = depth[A] + depth[B]。长度 ≥3 的环（比如 A→B→C→A）：A 的左侧是 C、右侧是 B，两个邻座全被环内的人占了，链上的人根本插不进来；整桌人数就等于环长。不同二元环的桌子是独立的圆桌，会议可以同时开多桌，所以 paired 用加法累加；而"一个大环"是另一种单独的结构，和"拼桌"方案互斥，最后取 max——把所有可能的结构都覆盖到了，答案必然在两者之中。

**longest_cycle 为什么每个环只算一次？** 第一个走到某环的起点会把整个环都标记 visited；后来的起点（比如例子里的 s=1）撞到 visited 的点就停，而那个点不在它自己的 position 里，所以不会重复统计。每个环由"第一次到达它的那次行走"独家计算。

**复杂度，用具体数字说：** 两个函数都是 O(n) 时间和空间。longest_cycle 里每个点至多被走一次（visited 挡住重复）；maximum_invitations 里剥离阶段每条边（每人恰一条出边，共 n 条）只被消一次，收环阶段每个点只进一次 cycle。LeetCode 2360/2127 的 n 上限是 10⁵，也就十万次量级的操作，跑起来瞬间完成，比一般图算法的 O(V+E) 还省心（这里 E 就是 n）。

## 8. 单元测试：每个用例在检查什么

```python
assert longest_cycle([3, 3, 4, 2, 3]) == 3
```
正常值：0 的链尾巴（0→3）后面挂着 3→2→4→3 的环，环长 3。这条同时验证"链段不计入环长"（position 差值恰好切掉尾巴）。

```python
assert longest_cycle([2, -1, 3, 1]) == -1
```
无环反例：0→2→3→1→(-1) 一条链走到黑，1 的出边是 -1。返回默认值 -1，验证 -1 终点的处理。

```python
assert longest_cycle([0]) == 1
```
自环边界：0→0 自己指自己，环长 1。（LeetCode 官方不允许自环，这是 “正确性与复杂度”部分 说的"教学自环"契约——本实现支持。）

```python
assert maximum_invitations([1, 0, 3, 2]) == 4
```
多桌拼接正常值：两个独立的二元环 {0,1} 和 {2,3}，无链可挂（depth 全 1），答案 1+1+1+1=4。验证 paired 的累加逻辑。

```python
def brute(f):
    n = len(f)
    best = 0
    for k in range(2, n + 1):
        for p in permutations(range(n), k):
            if p[0] != min(p):
                continue
            if all((f[p[i]] in (p[i - 1], p[(i + 1) % k]) for i in range(k))):
                best = k
                break
    return best

for n in range(2, 7):
    for _ in range(12):
        f = [rng.choice([j for j in range(n) if j != i]) for i in range(n)]
        assert maximum_invitations(f) == brute(f)
```
随机对拍：种子 91 固定可复现；对 2 到 6 人的每个规模随机生成 12 张"每人喜欢另一个别人"的函数图（`j!=i` 排除自环，符合契约）。暴力解 brute 枚举每种圆桌规模 k 和每种座次排列（`p[0]!=min(p)` 的 continue 是排列去重技巧——同一个环多种写法只算一次），检查"每人喜欢的人都是左右邻之一"（`f[p[i]] in (p[i-1],p[(i+1)%k])`，`%k` 处理圆桌首尾相接）。注意暴力枚举的是**单个圆桌**，它能和快速算法一致，是因为座位问题里"多桌拼"的答案在单个二元环情形下退化为桌内人数（brute 的 k 上限恰好覆盖 n——多个二元环拼到 n 人时那张"桌"按定义仍满足每人邻座是喜欢的人，两种理解殊途同归）。12×5=60 张随机图全对齐，正确性信心很足。

最后 `print('N091: 所有本章断言通过')` 供肉眼确认全绿。

In [5]:
assert longest_cycle([3, 3, 4, 2, 3]) == 3

assert longest_cycle([2, -1, 3, 1]) == -1

assert longest_cycle([0]) == 1

assert maximum_invitations([1, 0, 3, 2]) == 4

from itertools import permutations

from random import Random

rng = Random(91)

def brute(f):
    n = len(f)
    best = 0
    for k in range(2, n + 1):
        for p in permutations(range(n), k):
            if p[0] != min(p):
                continue
            if all((f[p[i]] in (p[i - 1], p[(i + 1) % k]) for i in range(k))):
                best = k
                break
    return best

for n in range(2, 7):
    for _ in range(12):
        f = [rng.choice([j for j in range(n) if j != i]) for i in range(n)]
        assert maximum_invitations(f) == brute(f)

print('N091: 所有本章断言通过')

N091: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 区分最大环与多个二元环加链的组合。

**练习 2：** 给出普通拓扑不能删除环的解释。

<details>
<summary>展开思路提示（不是完整答案）</summary>

**练习 1：区分"最大环"与"多个二元环加链的组合"。**
思路方向：构造两个输入——一个只有大环（如 favorite=[1,2,3,4,0] 的五元环，答案 5），一个全是二元环加长链（如 favorite=[1,0,3,2,0,4]，两个二元环各挂链）。提示：最能说明问题的是"大环人数 < 二元环拼桌人数"的对比，比如 3 人环 vs 两个带链的二元环（3 < 2+2+挂链），以及反过来的对比（5 人环 vs 两个光杆二元环）。手算每个例子的剥离过程和 depth 值，再用 `maximum_invitations` 验证。重点讲清楚为什么答案是 max 而不是 sum（两种结构不能同时要）。

**练习 2：解释普通拓扑排序为什么删不掉环。**
思路方向：拓扑排序每步删除"入度 0"的点，而环上每个点都有一条来自环内前驱的入边，这条边的起点永远不删除，所以入度永远到不了 0——环是拓扑排序的天生死角。提示：拿 favorite=[1,2,0]（三元环）手动模拟：degree=[1,1,1]，队列初始为空，剥离循环一步都不执行——这个"队列一开始就是空"的现象就是"删不掉环"的最直白证据。再对照本章代码说明这**正是**我们想要的：剥离剩环，环交给收环阶段单独处理。可以再用一个 4 点例子（三元环 + 一条入链）展示链被剥掉、degree 归零的过程。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
from collections import deque

def longest_cycle(edges):
    visited = [False] * len(edges)
    answer = -1
    for s in range(len(edges)):
        position = {}
        u = s
        while u != -1 and (not visited[u]):
            visited[u] = True
            position[u] = len(position)
            u = edges[u]
        if u in position:
            answer = max(answer, len(position) - position[u])
    return answer

def maximum_invitations(favorite):
    n = len(favorite)
    degree = [0] * n
    depth = [1] * n
    for v in favorite:
        degree[v] += 1
    q = deque((i for i, d in enumerate(degree) if d == 0))
    while q:
        u = q.popleft()
        v = favorite[u]
        depth[v] = max(depth[v], depth[u] + 1)
        degree[v] -= 1
        if degree[v] == 0:
            q.append(v)
    paired = best = 0
    seen = set()
    for s in range(n):
        if not degree[s] or s in seen:
            continue
        cycle = []
        u = s
        while u not in seen:
            seen.add(u)
            cycle.append(u)
            u = favorite[u]
        if len(cycle) == 2:
            paired += sum((depth[v] for v in cycle))
        else:
            best = max(best, len(cycle))
    return max(paired, best)

# 示例与回归测试
assert longest_cycle([3, 3, 4, 2, 3]) == 3

assert longest_cycle([2, -1, 3, 1]) == -1

assert longest_cycle([0]) == 1

assert maximum_invitations([1, 0, 3, 2]) == 4

from itertools import permutations

from random import Random

rng = Random(91)

def brute(f):
    n = len(f)
    best = 0
    for k in range(2, n + 1):
        for p in permutations(range(n), k):
            if p[0] != min(p):
                continue
            if all((f[p[i]] in (p[i - 1], p[(i + 1) % k]) for i in range(k))):
                best = k
                break
    return best

for n in range(2, 7):
    for _ in range(12):
        f = [rng.choice([j for j in range(n) if j != i]) for i in range(n)]
        assert maximum_invitations(f) == brute(f)

print('N091: 所有本章断言通过')

N091: 所有本章断言通过


## 11. 代表题与迁移

- **2360. Longest Cycle in a Graph（图中的最长环，canonical）**：这题就是本章 `longest_cycle` 的原始场景——函数图找最长环，考点是"本次路径 position + 全局 visited"双标记技巧，防止把链尾巴算进环长、防止旧路径重复计数。
- **2127. Maximum Employees to Be Invited to a Meeting（参加会议的最多员工数，canonical）**：这题就是本章 `maximum_invitations` 的原始场景——考点是入度剥离求 depth、二元环与长环的分类处理（二元环挂链可拼接、长环整环独占），是函数图结构分析的综合应用。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。